In [14]:
import pandas as pd
import unicodedata
import re

def normalize_name(name):
    # Strip diacritics (Şengün -> Sengun, Dončić -> Doncic)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    # Remove periods (A.J. -> AJ)
    name = name.replace('.', '')
    # Normalize suffixes: Sr., Jr., II, III, IV
    name = re.sub(r'\b(Sr|Jr|I{2,3}|IV)\b', '', name)
    # Collapse whitespace and lowercase
    name = ' '.join(name.lower().split())
    return name.strip()

In [15]:
estimated_wins_df_detailed = pd.read_csv('estimated_win.csv')
salaries_df = pd.read_csv('player_salaries.csv')

salaries_25_26 = salaries_df[['Player', '2025-26']]
estimated_wins_df = estimated_wins_df_detailed[['player_name', 'ewins', 'team_alias', 'pos_text', 'age', 'mpg']].rename(columns={'player_name': 'Player'})

# Create normalized join keys
estimated_wins_df['join_key'] = estimated_wins_df['Player'].apply(normalize_name)
salaries_25_26 = salaries_25_26.copy()
salaries_25_26['join_key'] = salaries_25_26['Player'].apply(normalize_name)

In [16]:
combined_df = pd.merge(estimated_wins_df, salaries_25_26.drop(columns=['Player']), on='join_key', how='inner')
combined_df['2025-26'] = combined_df['2025-26'].replace({'\$': '', ',': ''}, regex=True).astype(int)

combined_df['production_value'] = combined_df['ewins'] * 3_800_000
combined_df['diff'] = combined_df['production_value'] - combined_df['2025-26']

combined_df = combined_df.drop(columns=['join_key'])

In [17]:
combined_df.sort_values(by=['diff'], ascending=False).to_csv('production_value_difference.csv', index=False)

In [18]:
# Top 10 best value contracts (most production above salary)
print("=== TOP 10 BEST VALUE CONTRACTS ===")
top_value = combined_df.nlargest(10, 'diff')[['Player', 'team_alias', 'pos_text', 'ewins', '2025-26', 'production_value', 'diff']]
print(top_value.to_string(index=False))

print("\n=== TOP 10 MOST OVERPAID CONTRACTS ===")
worst_value = combined_df.nsmallest(10, 'diff')[['Player', 'team_alias', 'pos_text', 'ewins', '2025-26', 'production_value', 'diff']]
print(worst_value.to_string(index=False))

=== TOP 10 BEST VALUE CONTRACTS ===
                 Player team_alias pos_text    ewins  2025-26  production_value       diff
      Victor Wembanyama        SAS        C 13.48700 13376880        51250600.0 37873720.0
            Jalen Duren        DET        C  9.62928  6483144        36591264.0 30108120.0
       Collin Gillespie        PHX       PG  7.83112  2296274        29758256.0 27461982.0
           Kon Knueppel        CHA       SF  9.64945 10015680        36667910.0 26652230.0
Shai Gilgeous-Alexander        OKC       PG 16.96560 38333050        64469280.0 26136230.0
          Chet Holmgren        OKC        C 10.44490 13731368        39690620.0 25959252.0
          Amen Thompson        HOU       SG  9.37232  9690600        35614816.0 25924216.0
          Dyson Daniels        ATL       SG  8.23497  7707709        31292886.0 23585177.0
          Neemias Queta        BOS        C  6.77120  2349578        25730560.0 23380982.0
      Julian Champagnie        SAS       SF  6.62512  

In [19]:
# Team-level analysis: total surplus value (production_value - salary) per team
team_surplus = combined_df.groupby('team_alias').agg(
    total_salary=('2025-26', 'sum'),
    total_production_value=('production_value', 'sum'),
    total_surplus=('diff', 'sum'),
    num_players=('Player', 'count')
).sort_values('total_surplus', ascending=False)

print("=== TEAM SURPLUS VALUE (production value - salary) ===")
print(team_surplus.to_string())

=== TEAM SURPLUS VALUE (production value - salary) ===
            total_salary  total_production_value  total_surplus  num_players
team_alias                                                                  
OKC            179773002            2.282615e+08   4.848853e+07           14
SAS            184204928            2.089270e+08   2.472208e+07           17
DET            169872812            1.924409e+08   2.256811e+07           14
MIA            159804115            1.621124e+08   2.308274e+06           14
BOS            189348982            1.901325e+08   7.834894e+05           13
HOU            171960146            1.682678e+08  -3.692371e+06           14
CHA            179622853            1.715607e+08  -8.062200e+06           17
POR            154036459            1.310303e+08  -2.300620e+07           15
DEN            195921026            1.673664e+08  -2.855463e+07           16
CLE            221716257            1.898707e+08  -3.184561e+07           16
ATL            183728

In [20]:
# Cost per estimated win by position
pos_analysis = combined_df.groupby('pos_text').agg(
    avg_salary=('2025-26', 'mean'),
    avg_ewins=('ewins', 'mean'),
    avg_diff=('diff', 'mean'),
    num_players=('Player', 'count')
).sort_values('avg_diff', ascending=False)

pos_analysis['cost_per_ewin'] = (pos_analysis['avg_salary'] / pos_analysis['avg_ewins']).astype(int)

print("=== PRODUCTION VALUE BY POSITION ===")
print(pos_analysis.to_string())

=== PRODUCTION VALUE BY POSITION ===
            avg_salary  avg_ewins      avg_diff  num_players  cost_per_ewin
pos_text                                                                   
SF        9.467494e+06   1.914969 -2.190611e+06          102        4943940
C         1.100652e+07   2.282850 -2.331686e+06          107        4821392
SG        1.070429e+07   2.199310 -2.346906e+06          104        4867110
PG        1.389408e+07   2.819735 -3.179086e+06           76        4927441
PF        1.345287e+07   1.953084 -6.031150e+06           91        6888014


In [21]:
# Age curve: are younger or older players a better value?
combined_df['age_group'] = pd.cut(combined_df['age'], bins=[18, 22, 25, 28, 31, 35, 45],
                                   labels=['19-22', '23-25', '26-28', '29-31', '32-35', '36+'])

age_analysis = combined_df.groupby('age_group', observed=True).agg(
    avg_salary=('2025-26', 'mean'),
    avg_ewins=('ewins', 'mean'),
    avg_diff=('diff', 'mean'),
    num_players=('Player', 'count')
)

age_analysis['cost_per_ewin'] = (age_analysis['avg_salary'] / age_analysis['avg_ewins']).astype(int)

print("=== VALUE BY AGE GROUP ===")
print(age_analysis.to_string())

=== VALUE BY AGE GROUP ===
             avg_salary  avg_ewins      avg_diff  num_players  cost_per_ewin
age_group                                                                   
19-22      5.196256e+06   1.464287  3.680360e+05          101        3548658
23-25      8.601006e+06   2.076652 -7.097267e+05          143        4141764
26-28      1.476108e+07   2.926337 -3.641003e+06          100        5044218
29-31      1.783903e+07   2.824449 -7.106124e+06           70        6315932
32-35      1.536588e+07   1.657448 -9.067579e+06           46        9270807
36+        1.751759e+07   2.445429 -8.224964e+06           20        7163404


In [22]:
# Best value per team: each team's most underpaid player
best_per_team = combined_df.loc[combined_df.groupby('team_alias')['diff'].idxmax()]
best_per_team = best_per_team[['Player', 'team_alias', 'ewins', '2025-26', 'diff']].sort_values('diff', ascending=False)

print("=== BEST VALUE PLAYER PER TEAM ===")
print(best_per_team.to_string(index=False))

=== BEST VALUE PLAYER PER TEAM ===
                 Player team_alias     ewins  2025-26       diff
      Victor Wembanyama        SAS 13.487000 13376880 37873720.0
            Jalen Duren        DET  9.629280  6483144 30108120.0
       Collin Gillespie        PHX  7.831120  2296274 27461982.0
           Kon Knueppel        CHA  9.649450 10015680 26652230.0
Shai Gilgeous-Alexander        OKC 16.965600 38333050 26136230.0
          Amen Thompson        HOU  9.372320  9690600 25924216.0
          Dyson Daniels        ATL  8.234970  7707709 23585177.0
          Neemias Queta        BOS  6.771200  2349578 23380982.0
        Donovan Clingan        POR  7.690280  7178400 22044664.0
           Ryan Rollins        MIL  5.873200  4000000 18318160.0
       Donte DiVincenzo        MIN  7.590790 11990000 16855002.0
  Sandro Mamukelashvili        TOR  4.534710  2461463 14770435.0
           Jaylon Tyson        CLE  4.443470  3492480 13392706.0
             Saddiq Bey        NOP  5.016390  6118644 1

In [23]:
# Perform outer merge with indicator column
merged_df = pd.merge(estimated_wins_df, salaries_25_26.drop(columns=['Player']), on='join_key', how='outer', indicator=True)

# Find players missing from salaries_25_26 (present in estimated_wins_df but not in salaries)
missing_in_salaries = merged_df[merged_df['_merge'] == 'left_only']['Player']

# Find players missing from estimated_wins_df (present in salaries but not in estimated_wins_df)
missing_in_wins = merged_df[merged_df['_merge'] == 'right_only']['join_key']

# Print results
print("Players missing from salary data:")
print(missing_in_salaries.to_list())

print("\nPlayers missing from estimated wins data:")
print(missing_in_wins.to_list())

Players missing from salary data:
['A.J. Lawson', 'Alijah Martin', 'Andersson Garcia', 'Antonio Reeves', 'Bez Mbeng', 'Blake Hinson', 'Branden Carlson', 'Brooks Barnhizer', 'Caleb Love', 'Chaney Johnson', 'Chris Youngblood', 'Cody Martin', 'Daeqwon Plowden', 'David Jones Garcia', 'DeJon Jarreau', 'Drew Peterson', 'Dylan Cardwell', 'Egor Demin', 'Elijah Harkless', 'Ethan Thompson', 'Isaiah Crawford', 'Isaiah Livers', 'Jahmai Mashack', 'Jahmir Young', 'Jalen Slawson', 'Jamal Cain', 'Jamaree Bouyea', 'Jamir Watkins', 'Javon Small', 'Javonte Cooke', 'John Poulakidas', 'Johnny Juzang', 'Julian Reese', 'Kennedy Chandler', 'Kevin McCullar Jr.', 'KJ Simpson', 'Lachlan Olbrich', 'Leaky Black', 'LJ Cryer', 'Luke Travers', 'Malevy Leons', 'MarJon Beauchamp', 'Miles Kelly', 'Moussa Cisse', 'Nate Williams', 'Omer Yurtseven', 'Oscar Tshiebwe', 'Pete Nance', 'PJ Hall', 'Quenton Jackson', 'RayJ Dennis', 'Ron Harper Jr.', 'Ronald Holland II', 'Ryan Nembhard', 'Sharife Cooper', 'Taelon Peter', 'Taj Gibs

In [24]:
# Combine both lists into one DataFrame
missing_players_df = pd.DataFrame({'Player': pd.concat([missing_in_salaries, missing_in_wins]).unique()})

# Save to Excel
missing_players_df.to_excel('missing_players.xlsx', index=False)

# Display the first few rows
print(missing_players_df.head())

             Player
0       A.J. Lawson
1     Alijah Martin
2  Andersson Garcia
3    Antonio Reeves
4         Bez Mbeng
